# The Agentic Loop

In [2]:
from dotenv import load_dotenv
from ingest import load_search_index
from openai import OpenAI
import json
load_dotenv()

True

## Functions Define

In [3]:
def search(query):
    index = load_search_index()

    filter_dict={"course": "llm-zoomcamp"}
    boost_dict={"question": 2.0, "section": 0.5}

    search_resulte = index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )

    return search_resulte

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties":{
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            },
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    resulte_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output":resulte_json
    }

In [4]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## Make The Agentic Loop Manual

In [5]:
def agent_loop(question, instructions=instructions, model="gpt-5.4"):

    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    openai_client = OpenAI()

    while True:        
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
    
            elif item.type == "message":
                answer = item.content[0].text
        
        if has_function_calls == False:
            break

    return answer

## Send Requests

In [6]:
question = "I just discovered the course. Can I join it?"
print(agent_loop(question))

Yes — you can still join.

If you want a certificate, you need to submit your capstone project while the course is still accepting submissions. Also, certificates are only available with a live cohort; in self-paced mode, you can study the material, but project submission and peer review must happen during a live cohort.

Are there other areas you want to explore?
